# Exercise 08: Distributed Jacobi

**Names:** Tarik, Paul Dietze

Code used: `jacobi/jacobi_mpi.cc` and `jacobi/jacobi_omp.cc`.


## (b) Decomposition and halo exchange


For `n=1024`, `iterations=200`, sequential baseline `0.735 GUpdates/s`:

| ranks | blocking GUpdates/s | speedup | overlap GUpdates/s | speedup |
|---:|---:|---:|---:|---:|
| 1 | 0.326 | 0.44x | 0.173 | 0.24x |
| 2 | 0.252 | 0.34x | 0.301 | 0.41x |
| 4 | 0.933 | 1.27x | 1.034 | 1.41x |

The MPI result is correct. With only 1 or 2 ranks the MPI overhead is still too large, but with 4 ranks the overlapped version is faster than the sequential reference for this problem size.


## (c) Defect norm

Before computing the defect norm, we update the ghost rows again, because the top and bottom rows of each strip need neighbor values. Then each rank computes only its local sum of squared defects. These local sums are combined with `MPI_Allreduce`, and only then we take the square root.

Comparison with the sequential norm:

| run | MPI norm | sequential norm | difference |
|---|---:|---:|---:|
| `n=1024`, `iterations=200` | 1.23521 | 1.23521 | 2.66e-14 |
| `n=64`, `iterations=20` | 1.64555 | 1.64555 | 2.22e-16 |

Basically the same.


## (d) Overlap

Implemented the nonblocking version with `MPI_Isend` / `MPI_Irecv`. Inner rows are updated before waiting for the halo rows.

For `n=1024`, `iterations=200`:

| ranks | blocking | overlap |
|---:|---:|---:|
| 1 | 0.326 | 0.173 |
| 2 | 0.252 | 0.301 |
| 4 | 0.933 | 1.034 |

For 2 and 4 ranks, overlap is faster than the blocking version. For one rank it is slower, because there is basically no communication to hide and the nonblocking calls only add overhead.


## (e) Bonus: MPI+X

We implemented variant **(ii), with overlap**. MPI communication is done by the main thread, and OpenMP updates rows inside each rank.

We use:

```cpp
MPI_Init_thread(..., MPI_THREAD_FUNNELED, ...)
```

This is enough because only the main thread calls MPI.

| ranks | threads/rank | hybrid GUpdates/s |
|---:|---:|---:|
| 1 | 4 | 0.225 |
| 2 | 2 | 0.468 |
| 4 | 2 | 0.203 |

Clearly the `2 x 2` version produces the best result.


## (f) Comparison

Since the old OpenMP baseline was missing (because we did not do the previous exercise sheet 06), we implemented a simple row-parallel OpenMP Jacobi in `jacobi_omp.cc`. We compare pure OpenMP with pure MPI using the same total number of workers: 4.

| n | OpenMP, 4 threads | pure MPI overlap, 4 ranks |
|---:|---:|---:|
| 512 | 2.664 | 0.861 |
| 1024 | 1.152 | 1.034 |
| 2048 | 0.999 | 0.879 |
| 4096 | 0.954 | 0.987 |

![comparison](jacobi/comparison_plot.svg)

For small sizes OpenMP is clearly faster. This is expected because OpenMP threads can read neighbor rows directly from shared memory, while MPI has to exchange halo rows every iteration.

For larger sizes MPI gets closer, and for `n=4096` it is slightly faster in this run. The reason is the surface-to-volume effect: each MPI rank always communicates only about two boundary rows, but as the local strip gets larger it does much more computation between halo exchanges.

So the MPI overhead matters most for small strips. With larger problems, or on a real multi-node machine, MPI can become more competitive. The hybrid version from part (e) is also useful because it uses fewer MPI ranks and therefore fewer communicated strip boundaries.
